<a href="https://colab.research.google.com/github/zeynepdnnz/cs445-semeval-task5/blob/main/CS445_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
'''
homonym                                                     potential
judged_meaning      the difference in electrical charge between tw...
precontext          The old machine hummed in the corner of the wo...
sentence                          The potential couldn't be measured.
ending              She collected a battery reader and looked on e...
choices                                               [4, 5, 2, 3, 1]
average                                                           3.0
stdev                                                        1.581139
nonsensical                       [False, False, False, False, False]
sample_id                                                        1843
example_sentence         The circuit has a high potential difference.
'''

"\nhomonym                                                     potential\njudged_meaning      the difference in electrical charge between tw...\nprecontext          The old machine hummed in the corner of the wo...\nsentence                          The potential couldn't be measured.\nending              She collected a battery reader and looked on e...\nchoices                                               [4, 5, 2, 3, 1]\naverage                                                           3.0\nstdev                                                        1.581139\nnonsensical                       [False, False, False, False, False]\nsample_id                                                        1843\nexample_sentence         The circuit has a high potential difference.\n"

In [30]:
!pip install -q "transformers>=4.40,<4.45"

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Primary Dataset Fine-Tune Baseline**

In [32]:
import pandas
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, BatchEncoding,
    DataCollatorWithPadding, EvalPrediction,
    EarlyStoppingCallback
    )
from datasets import Dataset, DatasetDict
import numpy as np
from scipy.stats import spearmanr

In [33]:
ambistory_train_df = pandas.read_json("/content/train.json").T
ambistory_validation_df = pandas.read_json("/content/dev.json").T
ambistory_test_df = pandas.read_json("/content/test.json").T

In [34]:
model_id = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [35]:
def input_format(sample: pandas.Series) -> tuple[str, str]:
  story_part = [sample['precontext'], sample['sentence'], sample['ending'] or '']
  story_part = " ".join(story_part)
  meaning_part = (f"{sample['homonym']}: {sample['judged_meaning']} "
                   f"(e.g., \"{sample['example_sentence']}\")")

  return story_part, meaning_part


def tokenize_samples(batch: dict) -> BatchEncoding:
    story_parts = []
    meaning_parts = []
    for i in range(len(batch["precontext"])):
        ending = batch["ending"][i] or ''
        story = f"{batch['precontext'][i]} {batch['sentence'][i]} {ending}"
        meaning = (
            f"{batch['homonym'][i]}: {batch['judged_meaning'][i]} "
            f'(e.g., "{batch["example_sentence"][i]}")'
        )
        story_parts.append(story)
        meaning_parts.append(meaning)

    return tokenizer(
        story_parts,
        meaning_parts,
        truncation="only_first",
        max_length=256,
        padding=False,
    )


def metric_computation(eval_pred: EvalPrediction) -> dict[str, float]:
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    spearman_score, _ = spearmanr(predictions, labels)


    if np.isnan(spearman_score):
        spearman_score = 0.0

    mean_absolute_error = np.mean(np.abs(predictions - labels))
    root_mean_squared_error = np.sqrt(np.mean((predictions - labels) ** 2))

    return {
        "spearman": float(spearman_score),
        "mae": float(mean_absolute_error),
        "rmse": float(root_mean_squared_error),
    }
def compute_acc_within_std(trainer, dataset, original_df):
    preds = trainer.predict(dataset).predictions.squeeze()
    labels = original_df["average"].to_numpy(dtype=float)
    stdev = original_df["stdev"].to_numpy(dtype=float)

    threshold = np.maximum(stdev, 1.0)
    acc_within_std = float(np.mean(np.abs(preds - labels) <= threshold))

    return acc_within_std

In [36]:
from datasets import Dataset, DatasetDict, Value
import pandas as pd


ambistory_train_df["average"] = pd.to_numeric(
    ambistory_train_df["average"],
    errors="coerce"
)

ambistory_validation_df["average"] = pd.to_numeric(
    ambistory_validation_df["average"],
    errors="coerce"
)


ambistory_train_df = ambistory_train_df.dropna(subset=["average"]).reset_index(drop=True)
ambistory_validation_df = ambistory_validation_df.dropna(subset=["average"]).reset_index(drop=True)
ambistory_test_df = ambistory_test_df.drop(
    columns=["average", "stdev", "choices", "nonsensical"],
    errors="ignore"
).reset_index(drop=True)

raw_datasets = DatasetDict({
    "ambistory_train_set": Dataset.from_pandas(ambistory_train_df),
    "ambistory_validation_set": Dataset.from_pandas(ambistory_validation_df),
    "ambistory_test_set": Dataset.from_pandas(ambistory_test_df)
})


for split_name in ["ambistory_train_set", "ambistory_validation_set"]:
    raw_datasets[split_name] = raw_datasets[split_name].rename_column("average", "labels")
    raw_datasets[split_name] = raw_datasets[split_name].cast_column("labels", Value("float32"))

print(raw_datasets)
print(raw_datasets["ambistory_train_set"].column_names)
print(raw_datasets["ambistory_validation_set"].column_names)
print(raw_datasets["ambistory_test_set"].column_names)

Casting the dataset:   0%|          | 0/2280 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/588 [00:00<?, ? examples/s]

DatasetDict({
    ambistory_train_set: Dataset({
        features: ['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence'],
        num_rows: 2280
    })
    ambistory_validation_set: Dataset({
        features: ['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence'],
        num_rows: 588
    })
    ambistory_test_set: Dataset({
        features: ['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'sample_id', 'example_sentence'],
        num_rows: 930
    })
})
['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence']
['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence']
['homonym', 'judged_meaning', 'precontext', 'sentence'

In [37]:
def tokenize_dataset(dataset):
    cols_to_remove = [
        c for c in dataset.column_names
        if c != "labels"
    ]

    return dataset.map(
        tokenize_samples,
        batched=True,
        remove_columns=cols_to_remove
    )

tokenized_datasets = DatasetDict()

for split_name in raw_datasets:
    tokenized_datasets[split_name] = tokenize_dataset(raw_datasets[split_name])

Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

In [38]:
import json
import gc
import torch
import numpy as np
from itertools import product
from pathlib import Path


CONFIGS = [
    {"name": "E_short_warmup", "lr": 1e-5, "bs": 8, "epochs": 5, "warmup": 0.06},
]

SEEDS = [42, 1337, 2024]

from torch.optim import AdamW

def build_llrd_optimizer(model, base_lr=1e-5, layer_decay=0.85, weight_decay=0.01):
    """
    Layer-wise learning rate decay for DeBERTa.
    Upper layers get higher LR, lower layers get lower LR.
    Classifier/head gets base_lr.
    """

    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]


    layers = model.deberta.encoder.layer
    num_layers = len(layers)

    optimizer_grouped_parameters = []


    embedding_lr = base_lr * (layer_decay ** num_layers)

    embedding_params_decay = []
    embedding_params_no_decay = []

    for name, param in model.deberta.embeddings.named_parameters():
        full_name = f"deberta.embeddings.{name}"
        if not param.requires_grad:
            continue

        if any(nd in full_name for nd in no_decay):
            embedding_params_no_decay.append(param)
        else:
            embedding_params_decay.append(param)

    optimizer_grouped_parameters.append({
        "params": embedding_params_decay,
        "lr": embedding_lr,
        "weight_decay": weight_decay,
    })

    optimizer_grouped_parameters.append({
        "params": embedding_params_no_decay,
        "lr": embedding_lr,
        "weight_decay": 0.0,
    })




    for layer_idx, layer in enumerate(layers):
        layer_lr = base_lr * (layer_decay ** (num_layers - 1 - layer_idx))

        decay_params = []
        no_decay_params = []

        for name, param in layer.named_parameters():
            full_name = f"deberta.encoder.layer.{layer_idx}.{name}"
            if not param.requires_grad:
                continue

            if any(nd in full_name for nd in no_decay):
                no_decay_params.append(param)
            else:
                decay_params.append(param)

        optimizer_grouped_parameters.append({
            "params": decay_params,
            "lr": layer_lr,
            "weight_decay": weight_decay,
        })

        optimizer_grouped_parameters.append({
            "params": no_decay_params,
            "lr": layer_lr,
            "weight_decay": 0.0,
        })


    head_params_decay = []
    head_params_no_decay = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue


        if name.startswith("deberta.embeddings") or name.startswith("deberta.encoder"):
            continue

        if any(nd in name for nd in no_decay):
            head_params_no_decay.append(param)
        else:
            head_params_decay.append(param)

    optimizer_grouped_parameters.append({
        "params": head_params_decay,
        "lr": base_lr,
        "weight_decay": weight_decay,
    })

    optimizer_grouped_parameters.append({
        "params": head_params_no_decay,
        "lr": base_lr,
        "weight_decay": 0.0,
    })

    optimizer = AdamW(
        optimizer_grouped_parameters,
        lr=base_lr,
        eps=1e-8
    )

    return optimizer
def train_loop(config, seed):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=1,
        problem_type="regression",
        ignore_mismatched_sizes=True,
    )
    optimizer = build_llrd_optimizer(
      model,
      base_lr=config["lr"],
      layer_decay=0.85,
      weight_decay=0.01
    )

    args = TrainingArguments(
        output_dir=f"./sweep/{config['name']}_seed{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=config["lr"],
        per_device_train_batch_size=config["bs"],
        per_device_eval_batch_size=16,
        num_train_epochs=config["epochs"],
        weight_decay=0.01,
        warmup_ratio=config["warmup"],
        bf16=False,
        fp16=False,
        load_best_model_at_end=False,
        logging_steps=200,
        report_to="none",
        seed=seed,
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=data_collator,
        train_dataset=tokenized_datasets["ambistory_train_set"],
        eval_dataset=tokenized_datasets["ambistory_validation_set"],
        tokenizer=tokenizer,
        compute_metrics=metric_computation,
        optimizers=(optimizer, None),
    )

    trainer.train()

    eval_logs = [log for log in trainer.state.log_history if "eval_spearman" in log]
    best_eval = max(eval_logs, key=lambda x: x["eval_spearman"])

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "spearman": best_eval["eval_spearman"],
        "mae": best_eval["eval_mae"],
        "rmse": best_eval["eval_rmse"],
        "best_epoch": best_eval["epoch"],
    }


results = {}

for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"Config: {config['name']}  |  {config}")
    print('='*60)

    seed_results = []
    for seed in SEEDS:
        print(f"\n  seed={seed} ...", end=" ", flush=True)
        metrics = train_loop(config, seed)
        print(f"spearman={metrics['spearman']:.4f}  best_epoch={metrics['best_epoch']:.0f}")
        seed_results.append(metrics)

    spearmans = [r["spearman"] for r in seed_results]
    maes = [r["mae"] for r in seed_results]
    rmses = [r["rmse"] for r in seed_results]

    results[config["name"]] = {
        "config": config,
        "per_seed": seed_results,
        "spearman_mean": float(np.mean(spearmans)),
        "spearman_std": float(np.std(spearmans)),
        "mae_mean": float(np.mean(maes)),
        "rmse_mean": float(np.mean(rmses)),
    }

    print(f"\n  -> Spearman: {np.mean(spearmans):.4f} ± {np.std(spearmans):.4f}")
    print(f"  -> MAE:      {np.mean(maes):.4f}")
    print(f"  -> RMSE:     {np.mean(rmses):.4f}")

    Path("./sweep").mkdir(exist_ok=True)
    with open("./sweep/results.json", "w") as f:
        json.dump(results, f, indent=2)


print("\n\n" + "="*60)
print("SWEEP SUMMARY (sorted by mean dev Spearman)")
print("="*60)

ranked = sorted(results.items(), key=lambda x: -x[1]["spearman_mean"])
print(f"\n{'Config':<20} {'Spearman':<20} {'MAE':<10} {'RMSE':<10}")
print("-" * 60)
for name, r in ranked:
    spearman_str = f"{r['spearman_mean']:.4f} ± {r['spearman_std']:.4f}"
    print(f"{name:<20} {spearman_str:<20} {r['mae_mean']:<10.4f} {r['rmse_mean']:<10.4f}")

best = ranked[0]
print(f"\n Best config: {best[0]}")
print(f"   {best[1]['config']}")
print(f"   Dev Spearman: {best[1]['spearman_mean']:.4f} ± {best[1]['spearman_std']:.4f}")


Config: E_short_warmup  |  {'name': 'E_short_warmup', 'lr': 1e-05, 'bs': 8, 'epochs': 5, 'warmup': 0.06}

  seed=42 ... 

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 2.7052, 'grad_norm': 44.13285446166992, 'learning_rate': 1.8510141078904075e-07, 'epoch': 0.7017543859649122}
{'eval_loss': 1.169143795967102, 'eval_spearman': 0.5352789053638708, 'eval_mae': 0.8467332720756531, 'eval_rmse': 1.081269383430481, 'eval_runtime': 2.2759, 'eval_samples_per_second': 258.356, 'eval_steps_per_second': 16.257, 'epoch': 1.0}
{'loss': 0.8997, 'grad_norm': 18.746606826782227, 'learning_rate': 1.5488077229287082e-07, 'epoch': 1.4035087719298245}
{'eval_loss': 1.1729674339294434, 'eval_spearman': 0.5740522812529274, 'eval_mae': 0.8318364024162292, 'eval_rmse': 1.083036184310913, 'eval_runtime': 2.2897, 'eval_samples_per_second': 256.803, 'eval_steps_per_second': 16.159, 'epoch': 2.0}
{'loss': 0.8019, 'grad_norm': 35.63081741333008, 'learning_rate': 1.246601337967009e-07, 'epoch': 2.1052631578947367}
{'loss': 0.6126, 'grad_norm': 37.8189697265625, 'learning_rate': 9.443949530053098e-08, 'epoch': 2.807017543859649}
{'eval_loss': 1.2176611423492432, 'eval_spea

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 3.0056, 'grad_norm': 60.727699279785156, 'learning_rate': 1.8510141078904075e-07, 'epoch': 0.7017543859649122}
{'eval_loss': 1.1237767934799194, 'eval_spearman': 0.5366802559574539, 'eval_mae': 0.8208085298538208, 'eval_rmse': 1.0600833892822266, 'eval_runtime': 2.2888, 'eval_samples_per_second': 256.901, 'eval_steps_per_second': 16.166, 'epoch': 1.0}
{'loss': 0.8946, 'grad_norm': 41.9156494140625, 'learning_rate': 1.5488077229287082e-07, 'epoch': 1.4035087719298245}
{'eval_loss': 1.158437967300415, 'eval_spearman': 0.5822754479869546, 'eval_mae': 0.842934250831604, 'eval_rmse': 1.0763075351715088, 'eval_runtime': 2.3568, 'eval_samples_per_second': 249.486, 'eval_steps_per_second': 15.699, 'epoch': 2.0}
{'loss': 0.763, 'grad_norm': 35.41621017456055, 'learning_rate': 1.246601337967009e-07, 'epoch': 2.1052631578947367}
{'loss': 0.5834, 'grad_norm': 35.87051010131836, 'learning_rate': 9.443949530053098e-08, 'epoch': 2.807017543859649}
{'eval_loss': 1.0463234186172485, 'eval_spea

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 2.9462, 'grad_norm': 46.58456039428711, 'learning_rate': 1.8510141078904075e-07, 'epoch': 0.7017543859649122}
{'eval_loss': 1.1556079387664795, 'eval_spearman': 0.5298150662191857, 'eval_mae': 0.8373441100120544, 'eval_rmse': 1.074992060661316, 'eval_runtime': 2.2707, 'eval_samples_per_second': 258.945, 'eval_steps_per_second': 16.294, 'epoch': 1.0}
{'loss': 0.9752, 'grad_norm': 24.10358428955078, 'learning_rate': 1.5488077229287082e-07, 'epoch': 1.4035087719298245}
{'eval_loss': 1.1747205257415771, 'eval_spearman': 0.5783920714359488, 'eval_mae': 0.8179209232330322, 'eval_rmse': 1.0838452577590942, 'eval_runtime': 2.2921, 'eval_samples_per_second': 256.538, 'eval_steps_per_second': 16.143, 'epoch': 2.0}
{'loss': 0.7806, 'grad_norm': 50.77374267578125, 'learning_rate': 1.246601337967009e-07, 'epoch': 2.1052631578947367}
{'loss': 0.6535, 'grad_norm': 21.759620666503906, 'learning_rate': 9.443949530053098e-08, 'epoch': 2.807017543859649}
{'eval_loss': 1.2391506433486938, 'eval_s

In [39]:
best_name = ranked[0][0]
best_config = results[best_name]["config"]

print(f"\nTraining final model with config: {best_name}")
print(f"  {best_config}")

seed_results = results[best_name]["per_seed"]
mean_spearman = results[best_name]["spearman_mean"]
representative_idx = np.argmin([abs(r["spearman"] - mean_spearman) for r in seed_results])
final_seed = SEEDS[representative_idx]
print(f"  using seed={final_seed} (closest to mean)")

final_model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)
final_optimizer = build_llrd_optimizer(
    final_model,
    base_lr=best_config["lr"],
    layer_decay=0.85,
    weight_decay=0.01
)

final_args = TrainingArguments(
    output_dir="./baseline_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_config["lr"],
    per_device_train_batch_size=best_config["bs"],
    per_device_eval_batch_size=16,
    num_train_epochs=best_config["epochs"],
    weight_decay=0.01,
    warmup_ratio=best_config["warmup"],
    bf16=False,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=final_seed,
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["ambistory_train_set"],
    eval_dataset=tokenized_datasets["ambistory_validation_set"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    optimizers=(final_optimizer, None),
)

final_trainer.train()
dev_acc_within_std = compute_acc_within_std(
    final_trainer,
    tokenized_datasets["ambistory_validation_set"],
    ambistory_validation_df
)

print(f"Dev acc_within_std: {dev_acc_within_std:.4f}")

print("\n" + "="*60)
print("FINAL TEST PREDICTION")
print("="*60)

test_preds = final_trainer.predict(
    tokenized_datasets["ambistory_test_set"]
).predictions.squeeze()


test_preds = np.clip(test_preds, 1.0, 5.0)

submission_df = pd.DataFrame({
    "sample_id": ambistory_test_df["sample_id"].values,
    "prediction": test_preds
})

Path("./baseline_final").mkdir(exist_ok=True)
submission_df.to_csv("./baseline_final/submission.csv", index=False)

print(submission_df.head())
print("\nSaved predictions to ./baseline_final/submission.csv")

final_trainer.save_model("./baseline_final/best_model")
with open("./baseline_final/run_info.json", "w") as f:
    json.dump({
        "config": best_config,
        "seed": final_seed,
        "dev_spearman_mean": results[best_name]["spearman_mean"],
        "dev_spearman_std": results[best_name]["spearman_std"],
        "dev_acc_within_std": dev_acc_within_std,
        "note": "Test set labels are hidden, so acc_within_std was computed on dev set only."
    }, f, indent=2)

print(f"\nSaved to ./baseline_final/")


Training final model with config: E_short_warmup
  {'name': 'E_short_warmup', 'lr': 1e-05, 'bs': 8, 'epochs': 5, 'warmup': 0.06}
  using seed=2024 (closest to mean)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,1.114200,1.170210,0.529960,0.834822,1.081762
2,0.748300,1.169234,0.586114,0.817388,1.081311
3,0.506700,1.285525,0.599422,0.853399,1.133810
4,0.481200,1.228443,0.610627,0.839158,1.108351
5,0.492200,1.164114,0.610902,0.818216,1.078941


Dev acc_within_std: 0.7704

FINAL TEST PREDICTION


  sample_id  prediction
0      2017    4.875000
1      2018    2.046875
2      2019    4.750000
3      2020    3.234375
4      2021    4.812500

Saved predictions to ./baseline_final/submission.csv

Saved to ./baseline_final/
